**Nombre completo: Carlos alberto hernandez**

**Grupo: 002**

**Fecha:10/10/2026**

---

# Experimentación con diferentes parámetros
**Simulación de Sistemas · Sesión del 7 de octubre de 2026 · Primera hora**

Hoy no vamos a aprender un método nuevo: vamos a **mover las perillas** de los que ya tenemos.

El método de hoy es de tres pasos:
1. Cambio un parámetro.
2. **Predigo** qué va a pasar (escríbelo en un comentario *antes* de correr).
3. Corro y comparo contra lo esperado.

Solo usamos `random` y `math`. Ejemplo de hilo: la ventanilla del banco.

In [1]:
import math
import random

random.seed(7)   # misma semilla, misma serie

## Funciones base
`exponencial(media)` es la transformada inversa de la primera hora. Todo lo demás la reutiliza.

Fíjate en el `1 - u`: `random.random()` puede dar 0, y el logaritmo de 0 truena. `1 - u` nunca es 0.

In [ ]:
def exponencial(media):
    u = random.random()
    return -math.log(1 - u) * media


def entre_llegadas(lam):
    """lam = clientes por minuto. Devuelve minutos hasta el siguiente cliente."""
    return exponencial(1 / lam)


def promedio(f, n=20000):
    return sum(f() for _ in range(n)) / n

## Perilla 1: `lam` (llegadas por minuto) — inversa
**Predice:** si `lam` sube, ¿el tiempo entre llegadas sube o baja?

In [ ]:
# Mi predicción: ...

for lam in (0.1, 0.25, 0.5, 1.0):
    m = promedio(lambda: entre_llegadas(lam))
    print(f"lam={lam}: promedio={m:.2f}  esperado={1/lam:.2f}")

lam=0.1: promedio=9.96  esperado=10.00
lam=0.25: promedio=4.00  esperado=4.00
lam=0.5: promedio=1.98  esperado=2.00
lam=1.0: promedio=1.00  esperado=1.00


**Ahora tú:** prueba con `lam = 2`. ¿Qué esperas? Escríbelo antes de correr.

In [ ]:
# Mi predicción: ...
# Pista: copia el for de arriba, pero con lam = 2

## Perilla 2: `k` (etapas de servicio) — convolución
Un servicio de `k` etapas, cada una de unos 2 minutos. Es un pipeline: un `for` que suma lo que tarda cada etapa.

**Predice:** si paso de 1 a 8 etapas, ¿cuánto crece el promedio?

In [ ]:
def servicio_etapas(k, media_etapa=2.0):
    total = 0
    for _ in range(k):
        total += exponencial(media_etapa)
    return total


# Mi predicción: ...

for k in (1, 2, 4, 8):
    m = promedio(lambda: servicio_etapas(k))
    print(f"k={k}: promedio={m:.2f}  esperado={2*k}")

## Perilla 3: `p` (clientes rápidos) — composición
Con probabilidad `p` el cliente es rápido (unos 2 min); si no, es lento (unos 8 min). Es un `if` al azar.

**Predice:** si bajo `p`, ¿el promedio sube o baja?

In [ ]:
def servicio_mixto(p_rapido, media_rapido=2.0, media_lento=8.0):
    if random.random() < p_rapido:
        return exponencial(media_rapido)
    return exponencial(media_lento)


# Mi predicción: ...

for p in (0.9, 0.7, 0.5, 0.3):
    m = promedio(lambda: servicio_mixto(p))
    print(f"p={p}: promedio={m:.2f}  esperado={2*p + 8*(1-p):.2f}")

## Perilla 4: `mu` y `sigma` — transformación directa
`random.gauss` ya trae la receta (por dentro usa Box-Müller). Tú solo pasas el promedio y la dispersión.

El `max(0.0, ...)` evita tiempos negativos: nadie tarda menos de cero minutos.

**Predice:** con el mismo `mu`, si `sigma` pasa de 1 a 3, ¿qué cambia: el promedio o qué tan lejos quedan los extremos?

In [ ]:
def servicio_normal(mu, sigma):
    return max(0.0, random.gauss(mu, sigma))


# Mi predicción: ...

for mu, sigma in ((5, 1), (5, 3), (8, 1)):
    vals = [servicio_normal(mu, sigma) for _ in range(20000)]
    m = sum(vals) / len(vals)
    s = (sum((v - m) ** 2 for v in vals) / len(vals)) ** 0.5
    print(f"mu={mu} sigma={sigma}: media={m:.2f} desv={s:.2f}")

> **Ojo:** pedimos `sigma = 3` y la desviación sale un poco menor (≈ 2.9). Es por el `max`: los tiempos negativos se cortan en cero y eso le quita dispersión.

## Perilla 5: cuántas corridas
Una **corrida** es una ejecución completa; una **réplica** es repetirla con otra semilla.

`corrida_ventanilla` simula 10 clientes y regresa la espera promedio. Tres "equipos" hacen distinto número de corridas.

**Predice:** si tres equipos hacen **una sola** corrida, ¿les sale lo mismo?

In [ ]:
def corrida_ventanilla(n, lam, media_servicio):
    llegada = 0.0
    fin_anterior = 0.0
    esperas = []
    for i in range(n):
        if i > 0:
            llegada += entre_llegadas(lam)
        servicio = exponencial(media_servicio)
        inicio = max(llegada, fin_anterior)
        fin_anterior = inicio + servicio
        esperas.append(inicio - llegada)
    return sum(esperas) / n


# Mi predicción: ...

for n_corridas in (1, 5, 30, 1000):
    res = []
    for _ in range(3):  # tres "equipos" distintos
        esperas = [corrida_ventanilla(10, 0.25, 3.0) for _ in range(n_corridas)]
        res.append(sum(esperas) / n_corridas)
    print(n_corridas, [round(r, 2) for r in res])

Con 1 corrida los tres equipos obtienen respuestas muy distintas. Con 1000, coinciden.

**Para pensar:** si un equipo hizo 1 corrida y otro hizo 1000, ¿a cuál le entregarías el número para decidir si abrir una segunda ventanilla?

## Actividad 7.1 — tu turno
1. Elige **una** perilla (`lam`, `k`, `p`, o `mu` y `sigma`).
2. **Predice** qué pasa y escríbelo en un comentario.
3. Corre con **cuatro valores** distintos y compara contra tu predicción.
4. Entrega este notebook (`.ipynb`) en tu repo de GitHub; el link va en la plataforma.

In [3]:
# Tu experimento aquí
import math, random

random.seed(42)  # semilla fija para reproducibilidad

def exponencial(media):
    u = random.random()
    return -math.log(1 - u) * media

def servicio_mixto(p_rapido, media_rapido=2.0, media_lento=8.0):
    if random.random() < p_rapido:
        return exponencial(media_rapido)
    return exponencial(media_lento)

def promedio(f, n=20000):
    return sum(f() for _ in range(n)) / n

# Mi predicción:
# Si bajo p (menos clientes rápidos), el promedio de tiempo de servicio sube.

for p in (0.95, 0.75, 0.5, 0.1):
    m = promedio(lambda: servicio_mixto(p))
    esperado = 2 * p + 8 * (1 - p)
    print(f"p={p:.2f} : promedio={m:.2f} esperado={esperado:.2f}")


p=0.95 : promedio=2.29 esperado=2.30
p=0.75 : promedio=3.54 esperado=3.50
p=0.50 : promedio=4.96 esperado=5.00
p=0.10 : promedio=7.43 esperado=7.40
